# EDA Profile

Auto-EDA profiles (ydata-profiling), one HTML **and** PDF each (PDF via headless Chrome), regenerated on every run.

By default this profiles only the **raw** source tables and the purchased **EXT_SOURCE** scores, which is fast. Set `PROFILE_ENGINEERED = True` in the config cell to also profile the engineered feature tables and the assembled master (slow).

- `eda/eda-raw/` — the raw source tables
- `eda/eda-external/` — the purchased `EXT_SOURCE` scores
- `eda/eda-interim/` — engineered feature tables (only if `PROFILE_ENGINEERED`)
- `eda/eda-master/` — assembled modelling master (only if `PROFILE_ENGINEERED`)

Large tables sampled to 100k rows.

In [1]:
import sys; sys.path.append("..")
import warnings; warnings.filterwarnings("ignore")
import subprocess
import pandas as pd
from ydata_profiling import ProfileReport
from pathlib import Path

RAW = Path("../data/raw")
INTERIM = Path("../data/interim")
EDA = Path("../eda")
SAMPLE = 100_000
PROFILE_ENGINEERED = False   # True also profiles the engineered feature tables + master (slow)
CHROME = "/Applications/Google Chrome.app/Contents/MacOS/Google Chrome"

def report(df, name, outdir):
    outdir.mkdir(parents=True, exist_ok=True)
    if len(df) > SAMPLE:
        df = df.sample(SAMPLE, random_state=0)
    html = outdir / f"{name}.html"
    ProfileReport(df, title=name, minimal=True).to_file(html)
    subprocess.run([CHROME, "--headless=new", "--disable-gpu", "--no-pdf-header-footer",
                    "--virtual-time-budget=60000",
                    f"--print-to-pdf={html.with_suffix('.pdf')}", html.resolve().as_uri()],
                   check=True, capture_output=True)

## Raw

In [2]:
for name in ["application_train", "bureau", "bureau_balance", "previous_application",
             "POS_CASH_balance", "installments_payments", "credit_card_balance"]:
    report(pd.read_csv(RAW / f"{name}.csv"), name, EDA / "eda-raw")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 341.50it/s]


## External

The purchased `EXT_SOURCE` scores on their own — distributions, missingness, and relationship to default. These three columns are the subject of the bought-vs-built analysis.

In [3]:
report(pd.read_csv(RAW / "application_train.csv",
                   usecols=["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3", "TARGET"]),
       "external_scores", EDA / "eda-external")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 1068.34it/s]


## Interim

In [4]:
if PROFILE_ENGINEERED:
    for pq in sorted(INTERIM.glob("*.parquet")):
        if pq.name == "model_matrix.parquet":
            continue
        report(pd.read_parquet(pq).drop(columns="SK_ID_CURR", errors="ignore"), pq.stem, EDA / "eda-interim")

## Master

In [5]:
if PROFILE_ENGINEERED:
    from src.data import load_master
    X, y = load_master()
    report(X.assign(TARGET=y.values), "master", EDA / "eda-master")